# 2D Nonlinear Schrödinger Equation (NLSE)
## Split-Step Dynamics, Self-Focusing, and Soliton Collision

This notebook simulates the general 2D Nonlinear Schrödinger Equation using the pseudo-spectral `PDESolver` framework. Unlike the scalar wave equation (which describes a physical field $u$), the NLSE describes the **complex envelope** $\psi(x,y,t)$ of a narrow-band wave packet — governing phenomena from optical beam propagation in Kerr media to deep-water wave envelopes and Bose-Einstein condensates.

---

## 1. The Governing Equation

$$
i\partial_t\psi + \alpha\partial_x^2\psi + \beta\partial_y^2\psi + \gamma|\psi|^2\psi = 0
$$

* **$\alpha, \beta$ (Dispersion/Diffraction):** Govern the linear spreading of the envelope. If $\alpha$ and $\beta$ have the same sign, the medium is isotropic. If opposite, it is anisotropic (e.g., deep-water waves).
* **$\gamma$ (Nonlinearity):** The Kerr effect. If $\gamma > 0$, the medium is **focusing** (self-phase modulation pulls the beam together). If $\gamma < 0$, it is **defocusing**.

---

## 2. Reformulation for the Solver

Following the Kuramoto-Sivashinsky template, we rearrange into a first-order-in-time evolution equation:

$$
\partial_t\psi = i\alpha\partial_x^2\psi + i\beta\partial_y^2\psi + i\gamma|\psi|^2\psi
$$

In Fourier space, $\partial_x^2 \to -\xi^2$ and $\partial_y^2 \to -\eta^2$, so the linear operator becomes:

$$
\text{Linear symbol:} \quad -i(\alpha\xi^2 + \beta\eta^2)
$$

The equation in the solver's format:

$$
\partial_t\psi = \underbrace{\Psi_{\text{op}}\!\left(-i(\alpha\xi^2 + \beta\eta^2)\right)\psi}_{\text{Linear diffraction (Fourier space)}} + \underbrace{i\gamma|\psi|^2\psi}_{\text{Kerr nonlinearity (Physical space)}}
$$

---

## 3. Physical Phenomena

* **Self-Focusing Collapse:** A beam with power above the critical threshold $P_c$ counteracts diffraction and collapses to a singularity.
* **Spatial Solitons:** Exact balance between diffraction and nonlinearity produces stable localized packets.
* **Oblique Soliton Collision:** Two beams approaching at an angle interact nonlinearly — they can merge, repel, form bound states (soliton molecules), or shed dispersive radiation.

We initialize **two Gaussian beams on an oblique collision course** to witness this rich nonlinear choreography.

# Implementation
## 0. Imports

In [ ]:
from solver import PDESolver, psiOp
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── NLSE Coefficients ──
ALPHA =  1.0   # x-diffraction/dispersion
BETA  =  1.0   # y-diffraction/dispersion (same sign → isotropic medium)
GAMMA =  -2.0   # Nonlinearity (positive → focusing Kerr effect)

# ── Initial Beam Parameters ──
A0 = 1.8       # Peak amplitude (above critical power → self-focusing)
W0 = 1.5       # Beam waist (width)
X0 = 6.0       # Initial x-separation from origin
Y0 = 3.0       # Initial y-separation from origin
KX = 0.25       # x-momentum (controls approach speed: v_gx = 2·α·KX)
KY = 0.5       # y-momentum (oblique angle: v_gy = 2·β·KY)

# ── Grid and Time ──
# Large domain to prevent boundary interactions during collision
Lx, Ly   = 10.0, 10.0
# Nx, Ny = 32, 32 
# Nx, Ny = 64, 64    
# Nx, Ny = 128, 128    
Nx, Ny = 256, 256    
# Nx, Ny = 512, 512   

Lt, Nt   = 10.0, 800
# Lt, Nt   = 30.0, 600
# Lt, Nt   = 40.0, 800
# Lt, Nt   = 50.0, 1000
# Lt, Nt   = 60.0, 1200
n_frames = 250

## 2. Grid setup

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol

In [ ]:
t, x, y   = sp.symbols('t x y', real=True)
xi, eta   = sp.symbols('xi eta', real=True)
psi_func  = sp.Function('psi')
psi_field = psi_func(t, x, y)

# ── Linear symbol in Fourier space ──
# From: ∂ψ/∂t = iα·∂²ψ/∂x² + iβ·∂²ψ/∂y² + ...
# Fourier: ∂²/∂x² → -ξ²,  ∂²/∂y² → -η²
# So: iα(-ξ²) + iβ(-η²) = -i(αξ² + βη²)

symbol_linear = -sp.I * (ALPHA * xi**2 + BETA * eta**2)

print("Principal symbol (linear part):")
print("  a(ξ, η) =", symbol_linear)

## 4. NLSE equation

In [ ]:
# ∂ψ/∂t = psiOp(-i(αξ² + βη²), ψ)     +  iγ|ψ|²ψ
#        ─────────────────────────      ───────────
#        Linear diffraction (Fourier)   Kerr nonlinearity (Physical)

equation = sp.Eq(
    sp.diff(psi_field, t),
    psiOp(symbol_linear, psi_field)
    + sp.I * GAMMA * sp.Abs(psi_field)**2 * psi_field
)

print("NLSE Equation:")
print("  ∂ψ/∂t = psiOp(-i(αξ² + βη²), ψ) + iγ|ψ|²ψ")

## 5. Initial conditions: Oblique soliton collision

In [ ]:
def initial_condition_nlse(xx, yy):
    """
    Two Gaussian beams on an oblique collision course.
    Beam 1: starts at (-X0, -Y0), moves toward (+x, +y)
    Beam 2: starts at (+X0, +Y0), moves toward (-x, -y)
    They meet at the origin at t ≈ X0 / (2·α·KX) = 6 / 2 = 3.0
    """
    # Beam 1
    env1 = np.exp(-((xx + X0)**2 + (yy + Y0)**2) / W0**2)
    phase1 = KX * xx + KY * yy
    beam1 = A0 * env1 * np.exp(1j * phase1)
    
    # Beam 2 (mirror image, opposite momentum)
    env2 = np.exp(-((xx - X0)**2 + (yy - Y0)**2) / W0**2)
    phase2 = -(KX * xx + KY * yy)
    beam2 = A0 * env2 * np.exp(1j * phase2)
    
    return beam1 + beam2

## 6. Solver setup

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet',
    initial_condition=initial_condition_nlse,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve

In [ ]:
frames = solver.solve()

## 8. Visualization

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='abs',    # Show |ψ| (envelope amplitude)
    overlay=None,  # Contours reveal the phase structure and interference fringes
    mode='surface',
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
ani.save('nlse_soliton_collision.mp4', writer='ffmpeg', fps=20, dpi=100)
print("✅ Saved to nlse_soliton_collision.mp4")